In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import pandas as pd 
import diptest

In [ ]:

path = f"/home/bekah/m3-pipeline-dev/l0_l1b_l2/example_data/m3g20081226t073624_l1b_rdn.fits" 
path = f"/home/bekah/m3-pipeline-dev/l0_l1b_l2/example_data/m3g20081130t034731_l0.fits"
with fits.open(path) as hdul:
    dark = hdul[0].data.transpose(1, 0, 2)

#sub_dark = dark - np.median(dark, axis=0) 

    

In [ ]:
sub_dark = sub_dark.transpose(1, 0, 2) 

In [ ]:

path = f"/home/bekah/m3-pipeline-dev/data/l1b/m3g20090601t062753_l1b_rdn.fits" 
#path = f"/home/bekah/m3-pipeline-dev/data/l0_T/m3t20090516t123035_l0.fits" 
#path =  "/home/bekah/m3-pipeline-dev/l0_l1b_l2/example_data/m3g20081130t050351_l1b_rdn.fits"
with fits.open(path) as hdul:
    obs = hdul[0].data
    

In [ ]:

band = 19
col =  25
col_a = col - 1
col_c = col + 1


In [ ]:
x = np.linspace(0, 10, 25)
y = x  # Since x = y

col_a = col - 1
col_c = col + 1

plt.scatter(obs[band, :, col],obs[band, :, col_a],  alpha=.05)
plt.scatter(obs[band, :, col],obs[band, :, col_c],  alpha=.05)

plt.xlabel("bad col")
plt.ylabel("side cols") 
plt.plot(x, y, label="y = x", color="blue")


In [ ]:
plt.scatter(obs[band, :, col]-obs[band, :, col_a], obs[band, :, col]-obs[band, :, col_c], alpha=.1)

In [ ]:
plt.scatter(obs[band, :, col]/obs[band, :, col_a], obs[band, :, col]/obs[band, :, col_c], alpha=.1)

In [ ]:
# Authors: The scikit-learn developers
# SPDX-License-Identifier: BSD-3-Clause

import itertools

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from scipy import linalg

from sklearn import mixture

color_iter = itertools.cycle(["navy", "c", "cornflowerblue", "gold", "darkorange"])


def plot_results(X, Y_, means, covariances, index, title):
    splot = plt.subplot(2, 1, 1 + index)
    for i, (mean, covar, color) in enumerate(zip(means, covariances, color_iter)):
        v, w = linalg.eigh(covar)
        v = 2.0 * np.sqrt(2.0) * np.sqrt(v)
        u = w[0] / linalg.norm(w[0])
        # as the DP will not use every component it has access to
        # unless it needs it, we shouldn't plot the redundant
        # components.
        if not np.any(Y_ == i):
            continue
        plt.scatter(X[Y_ == i, 0], X[Y_ == i, 1], 0.8, color=color)

        # Plot an ellipse to show the Gaussian component
        angle = np.arctan(u[1] / u[0])
        angle = 180.0 * angle / np.pi  # convert to degrees
        ell = mpl.patches.Ellipse(mean, v[0], v[1], angle=180.0 + angle, color=color)
        ell.set_clip_box(splot.bbox)
        ell.set_alpha(0.5)
        splot.add_artist(ell)

    plt.xlim(-9.0, 5.0)
    plt.ylim(-3.0, 6.0)
    plt.xticks(())
    plt.yticks(())
    plt.title(title)


# Number of samples per component

# Generate random sample, two components
X = np.c_[obs[band, :, col]-obs[band, :, col_a], obs[band, :, col]-obs[band, :, col_c]]

# Fit a Gaussian mixture with EM using five components
gmm = mixture.GaussianMixture(n_components=2, covariance_type="full").fit(X)
plot_results(X, gmm.predict(X), gmm.means_, gmm.covariances_, 0, "Gaussian Mixture")

# Fit a Dirichlet process Gaussian mixture using five components
dpgmm = mixture.BayesianGaussianMixture(n_components=2, covariance_type="full").fit(X)
plot_results(
    X,
    dpgmm.predict(X),
    dpgmm.means_,
    dpgmm.covariances_,
    1,
    "Bayesian Gaussian Mixture with a Dirichlet process prior",
)

plt.show()

In [ ]:
gmm.means_

In [ ]:
plt.hist(obs[band, :, col]-obs[band, :, col_a], bins=50, alpha=.5)
plt.hist(obs[band, :, col]-obs[band, :, col_c], bins=50, alpha=.5)
plt.axvline(0, color='b')
plt.axvline(np.mean(obs[band, :, col]-obs[band, :, col_c]), color='r')
plt.axvline(np.mean(obs[band, :, col]-obs[band, :, col_a]), color='g')
plt.axvline(np.median(obs[band, :, col]-obs[band, :, col_c]), color='r', ls=':')
plt.axvline(np.median(obs[band, :, col]-obs[band, :, col_a]), color='g', ls=':')
plt.xlabel("diff, main col - side col") 

In [ ]:
plt.hist(obs[band, :, col]/obs[band, :, col_a], bins=50, alpha=.5)
plt.hist(obs[band, :, col]/obs[band, :, col_c], bins=50, alpha=.5)
plt.axvline(1, color='b')
plt.axvline(np.mean(obs[band, :, col]/obs[band, :, col_c]), color='r')
plt.axvline(np.mean(obs[band, :, col]/obs[band, :, col_a]), color='g')
plt.axvline(np.median(obs[band, :, col]/obs[band, :, col_c]), color='r', ls=':', lw=2)
plt.axvline(np.median(obs[band, :, col]/obs[band, :, col_a]), color='g', ls=':')
plt.xlabel("diff, main col - side col") 

In [ ]:
obs.shape

In [ ]:
fits.writeto(
                f"/home/bekah/m3-pipeline-dev/outputs/l1b_sorted.fits", 
                -np.sort(-obs, axis=1), 
                overwrite=True
                )

In [ ]:
obs[:,:,0:303].shape

In [ ]:
obs[:,:,1:304]

In [ ]:
fits.writeto(
                f"/home/bekah/m3-pipeline-dev/outputs/l1b_diff.fits", 
               ((obs[:,:,4:300]-obs[:,:,3:299])+(obs[:,:,4:300]-obs[:,:,5:301]))/2, 
                overwrite=True
                )

In [ ]:
sort = -np.sort(-obs, axis=1)

In [ ]:
sort.shape

In [ ]:
plt.scatter(range(304), sort[45, 5000, :]/np.median(sort[45, 5000, :]), s=.5)
plt.scatter(range(304), sort[45, 4600, :]/np.median(sort[45, 4600, :]), s=.5)
plt.scatter(range(304), sort[45, 3400, :]/np.median(sort[45, 3400, :]), s=.5)
plt.scatter(range(304), sort[45, 6194, :]/np.median(sort[45, 6194, :]), s=.5)
# plt.scatter(range(304), sort[45, -100, :]/np.median(sort[45, -100, :]), s=.5)
# plt.scatter(range(304), sort[45, 100, :]/np.median(sort[45, 100, :]), s=.5)

In [ ]:
from scipy.stats import kurtosis

#data = obs[band, :, col]/obs[band, :, col_a]
data = obs[band, :, 249]-obs[band, :, 248]

excess_kurt = kurtosis(data)
print("Excess Kurtosis:", excess_kurt)

# Calculate Pearson's kurtosis (Normal distribution == 3)
pearson_kurt = kurtosis(data, fisher=False)
print("Pearson Kurtosis:", pearson_kurt)


In [ ]:
np.std(obs[band, :, col])

In [ ]:
np.std(obs[band, :, col_a])

In [ ]:
plt.scatter(obs[75, :, 156]-obs[75, :, 155], obs[75, :, 156]-obs[75, :, 157],s=1, alpha=.1)
plt.axvline(x=0)
plt.axhline(y=0)

In [ ]:
path = f"/home/bekah/m3-pipeline-dev/outputs/m3g20090815t083731_back.fits" 

with fits.open(path) as hdul:
    back = hdul[0].data

In [ ]:
path = f"/home/bekah/m3-pipeline-dev/outputs/m3g20090815t083731_full.fits" 

with fits.open(path) as hdul:
    full = hdul[0].data

In [ ]:
full.shape

In [ ]:
plt.scatter(range(85),full[:, 1300, 156], s=1)
plt.errorbar(range(85),back[:, 156])

In [ ]:
plt.errorbar(range(85), full[:, 1700, 140], yerr=back[:, 140], fmt='o', markersize=3)

In [ ]:
plt.scatter(range(len(dark[54, :, 2])), dark[54, :, 2], s=1) 
plt.scatter(range(len(obs[54, :, 2])), obs[54, :, 2], s=1) 

In [ ]:
np.std(dark[74, :, 3]) 

In [ ]:
dark[9, :, 3].shape

In [ ]:
np.std(obs[74, 900:1178, 3]) 

In [ ]:
plt.hist(obs[9, :, 3], bins=50)
plt.hist(dark[9, :, 3], bins=50)

In [ ]:
dark.shape

In [ ]:
for i in range(260): 
    plt.scatter(i, np.std(dark[i, 10:, 3], axis=0)/np.std(obs[i, 100:593, 3], axis=0), alpha=.4) 
    plt.scatter(i, np.std(dark[i, 10:, 639], axis=0)/np.std(obs[i, 100:593, 639], axis=0), alpha=.4)
    plt.scatter(i, np.std(dark[i, 10:, 2], axis=0)/np.std(obs[i, 100:593, 2], axis=0), alpha=.4)
    plt.scatter(i, np.std(dark[i, 10:, 638], axis=0)/np.std(obs[i, 100:593, 638], axis=0), alpha=.4)
plt.xlabel("band, target") 
plt.ylabel("std(dark) dark cols/std(obs) dark cols") 

In [ ]:
for i in range(260): 
    plt.scatter(i, np.std(dark[i, 10:, 3], axis=0)-np.std(obs[i, 100:593, 3], axis=0), alpha=.4) 
    plt.scatter(i, np.std(dark[i, 10:, 639], axis=0)-np.std(obs[i, 100:593, 639], axis=0), alpha=.4)
    plt.scatter(i, np.std(dark[i, 10:, 2], axis=0)-np.std(obs[i, 100:593, 2], axis=0), alpha=.4)
    plt.scatter(i, np.std(dark[i, 10:, 638], axis=0)-np.std(obs[i, 100:593, 638], axis=0), alpha=.4)
plt.xlabel("band, target") 
plt.ylabel("std(dark) dark cols - std(obs) dark cols")

In [ ]:
for i in range(260): 
    plt.scatter(i, np.std(dark[i, :, 3], axis=0), alpha=.4, c='orange') 
    plt.scatter(i, np.std(obs[i, 100:593, 3], axis=0), alpha=.4, c='cyan') 
    
    plt.scatter(i, np.std(dark[i, :, 120], axis=0), alpha=.4, c='orange') 

    plt.scatter(i, np.std(dark[i, :, 638], axis=0), alpha=.4, c='green') 
    plt.scatter(i, np.std(obs[i, 100:593, 638], axis=0), alpha=.4, c='red') 

In [ ]:
for i in range(85): 
    plt.scatter(i, np.std(dark[i, :, 120], axis=0)/np.std(obs[i, 900:1178, 120], axis=0), alpha=.4) 
    plt.scatter(i, np.std(dark[i, :, 210], axis=0)/np.std(obs[i, 900:1178, 210], axis=0), alpha=.4)
    plt.scatter(i, np.std(dark[i, :, 42], axis=0)/np.std(obs[i, 900:1178, 42], axis=0), alpha=.4)
    plt.scatter(i, np.std(dark[i, :, 12], axis=0)/np.std(obs[i, 900:1178, 12], axis=0), alpha=.4)


In [ ]:
dark.shape

In [ ]:
for i in range(85): 
    plt.scatter(i, np.mean(dark[i, :, 3]), alpha=.4) 
    plt.scatter(i, np.mean(dark[i, :, 4]), alpha=.4)
    plt.scatter(i, np.mean(dark[i, :, 2]), alpha=.4)
    plt.scatter(i, np.mean(dark[i, :, 4]), alpha=.4)

In [ ]:
from scipy.stats import iqr

for i in range(85):
    pairs = [
        (dark[i, :, 3], obs[i, :, 3]),
        (dark[i, :, 4], obs[i, :, 318]),
        (dark[i, :, 2], obs[i, :, 2]),
        (dark[i, :, 4], obs[i, :, 319]),
    ]

    for d, o in pairs:
        d_med, o_med = np.median(d), np.median(o)
        d_iqr, o_iqr = iqr(d), iqr(o)

    plt.scatter(i, d_iqr/o_iqr)